# Somo la 08 - Mfano wa Ubunifu wa Wakala Wengi


## Usanidi


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kwa Nini Mifumo ya Wakala Wengi?

Kazi za ulimwengu halisi kama kupanga safari zinahusisha aina nyingi tofauti za utaalamu — usafirishaji, maarifa ya eneo, upangaji bajeti, na zaidi. Wakala mmoja anayejaribu kushughulikia kila kitu haraka huwa mgumu kudhibiti.

Mifumo ya wakala wengi husuluhisha hili kupitia **utaalamu maalum**: kila wakala anazingatia eneo moja la utaalamu, akitoa matokeo bora zaidi kuliko mtu anayejumuisha mambo yote. Pia huboresha **uwezo wa kupanuka** — unaweza kuongeza mawakala wapya (mfano, mtaalamu wa ndege, mkosoaji wa mikahawa) bila kuandika upya mchakato uliopo. Mawakala huunganishwa pamoja kupitia njia iliyo pangwa, wakipitisha muktadha kutoka kwa mmoja hadi mwingine.


## Kuunda Wakala Maalum


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Kujenga Mtiririko wa Kazi wa Mfululizo

`WorkflowBuilder` inakuwezesha kuunganisha maajenti katika grafu iliyoratibiwa. Hapa tunaunda bomba rahisi la hatua mbili: **TravelPlanner** huunda ratiba, kisha **TravelConcierge** hurejea na kuiboresha.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kuongeza Wakala Zaidi kwenye Mtiririko wa Kazi

Mmoja wa faida kubwa za muundo wa mawakala wengi ni jinsi unavyoweza kupanuliwa kwa urahisi. Hapo chini tunaongeza wakala **BudgetReviewer** ambaye anakagua mpango dhidi ya bajeti ya msafiri, anaangazia vitu vinavyoweza kusababisha gharama kuzidi kikomo, na kupendekeza mbadala za kuokoa pesa. Mtiririko wa kazi sasa unafanya kazi kwa mawakala watatu kwa mfuatano:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kuunda mawakala maalum** — kila mmoja akiwa na jukumu la kuzingatia (mpango, msaidizi, ukaguzi wa bajeti).
2. **Kuunganisha mawakala katika mtiririko wa kazi mfululizo** kwa kutumia `WorkflowBuilder` na `add_edge`.
3. **Kusambaza matokeo** kutoka kwenye mtiririko wa mawakala wengi, ukifuatilia ni wakala gani anazungumza.
4. **Kupanua mtiririko wa kazi** kwa kuongeza mawakala wapya katika mnyororo bila kubadilisha waliopo.

Mfano wa muundo wa mawakala wengi unahakikisha kila wakala ni rahisi huku ukizalisha matokeo tajiri zaidi na yaliyochunguzwa vizuri kuliko yatakavyozalishwa na wakala mmoja peke yake.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
